
# Drift Studio — RUN + Benchmark + Compare (todo en uno, sin artefactos)

Este notebook **fusiona** el *Benchmark* y el *Methods Compare* en un solo flujo:
- Ejecuta el pipeline (prioriza funciones **en memoria** desde `drift_funcs.py`; si no existen, usa un **run temporal** y limpia).
- Construye tablas **summary** (por estrategia/métrica) y **columns** (por variable) **sin** guardar imágenes.
- Muestra **subplots inline** (heatmap de drift, matrices de correlación) y tablas: *top columnas*, *p95/p50*, *estabilidad*, y *selección*.
- **No** deja archivos extra (a menos que elijas exportar explícitamente).


## 1) Setup

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil, tempfile, os

# === Config mínima ===
PLANTS = ['planta1','planta2','planta3']
STRATEGIES = ['decay','golden','seasonal']
METRICS = ['evidently_default','ks','mannwhitney','psi','wasserstein']

# Rutas de entrada (ajusta a tus CSV crudos)
plant_files = {
    'planta1': Path('../df_procesados/df_planta_1.csv'),
    'planta2': Path('../df_procesados/df_planta_2.csv'),
    'planta3': Path('../df_procesados/df_planta_3.csv'),
}
flag_files = {
    'planta1': Path('../df_procesados/flags_p1.csv'),
    'planta2': Path('../df_procesados/flags_p2.csv'),
    'planta3': Path('../df_procesados/flags_p3.csv'),
}

# Parámetros del pipeline (usamos los tuyos por defecto)
PARAMS = dict(
    CURRENT_WINDOW='3D',
    RESAMPLE=None,
    RESAMPLE_AGG='mean',
    EXCLUDE_COLUMNS=['pH Ecualizador 2 (Tk 250m3)', "Conductividad DAF", "Temperatura DAF", 
                     'pH entrada a Ecualizador 1', "Flujo Aire Reactor 1", "OD Reactor 1"],
    NUM_METHOD='auto',
    NUM_THRESHOLD=None,
    DECAY_HALF_LIFE_HOURS=24*7,
    DECAY_WEIGHT_MASS=0.95,
    GOLDEN_WIN='30min', GOLDEN_STEP='10min', GOLDEN_K=40,
    SEASONAL_WEEKS_BACK=12,
    SAVE_HTML=False
)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)


## 2) Ejecución preferentemente **en memoria**

In [ ]:

# Intentamos usar una API en memoria si existe en drift_funcs.py
in_memory_ok = False
df_sum_all = pd.DataFrame()
df_cols_all = pd.DataFrame()

try:
    from drift_funcs import run_drift_memory  # función opcional que devuelve (df_sum_all, df_cols_all)
    df_sum_all, df_cols_all = run_drift_memory(
        plant_names=PLANTS,
        strategies=STRATEGIES,
        plant_files=plant_files,
        flag_files=flag_files,
        **PARAMS
    )
    in_memory_ok = True
    print("[OK] Usando API en memoria: run_drift_memory")
except Exception as e:
    print("[INFO] API en memoria no disponible o falló:", e)
    in_memory_ok = False

df_sum_all.shape, df_cols_all.shape


## 3) Fallback limpio (run temporal → leer CSV → limpiar)

In [ ]:

if not in_memory_ok:
    from drift_funcs import run_drift_batch  # tu función existente
    tmpdir = Path(tempfile.mkdtemp(prefix='drift_tmp_'))
    print("[TMP] Creando ejecución temporal en:", tmpdir)

    paths, errs = run_drift_batch(
        plant_names=PLANTS,
        strategies=STRATEGIES,
        plant_files=plant_files,
        flag_files=flag_files,
        output_root=tmpdir,
        **PARAMS
    )
    # Recolectar summary/columns por planta
    sum_list, col_list = [], []
    for plant in PLANTS:
        s = tmpdir / plant / "_comparisons" / f"{plant}_summary_all_metrics.csv"
        c = tmpdir / plant / "_comparisons" / f"{plant}_columns_all_metrics.csv"
        if s.exists():
            sum_list.append(pd.read_csv(s).assign(plant=plant))
        if c.exists():
            col_list.append(pd.read_csv(c).assign(plant=plant))

    df_sum_all = pd.concat(sum_list, ignore_index=True) if sum_list else pd.DataFrame()
    df_cols_all = pd.concat(col_list, ignore_index=True) if col_list else pd.DataFrame()

    # Limpiar ejecución temporal
    try:
        shutil.rmtree(tmpdir, ignore_errors=True)
        print("[TMP] Limpieza OK")
    except Exception as e:
        print("[WARN] No pude borrar tmp:", e)

print("df_sum_all:", df_sum_all.shape)
print("df_cols_all:", df_cols_all.shape)
display(df_sum_all.head(10)); display(df_cols_all.head(10))


## 4) Dashboards inline (sin archivos)

In [ ]:

def heatmap_drift_rate_inline(df_summary, plant):
    sub = df_summary.query("plant == @plant").copy()
    if sub.empty:
        print(f"[{plant}] Sin datos para heatmap"); return
    sub['strategy'] = pd.Categorical(sub['strategy'], categories=STRATEGIES, ordered=True)
    sub['metric'] = pd.Categorical(sub['metric'], categories=METRICS, ordered=True)
    pv = sub.pivot_table(index='strategy', columns='metric', values='drift_rate_pct', aggfunc='mean').sort_index().reindex(columns=METRICS)
    fig, ax = plt.subplots(1,1, figsize=(8,4))
    im = ax.imshow(pv.values, aspect='auto')
    ax.set_yticks(range(pv.shape[0])); ax.set_yticklabels(pv.index)
    ax.set_xticks(range(pv.shape[1])); ax.set_xticklabels(pv.columns, rotation=30, ha='right')
    ax.set_title(f"Drift rate (%) — {plant}")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

def corr_matrices_inline(df_cols, plant):
    fig, axes = plt.subplots(1, 3, figsize=(12,3.6))
    last_im = None
    for i, strat in enumerate(STRATEGIES):
        sub = df_cols.query("plant == @plant and strategy == @strat")
        if sub.empty:
            axes[i].set_title(f"{strat}: sin datos"); axes[i].axis("off"); continue
        piv = sub.pivot_table(index='col', columns='metric', values='score', aggfunc='mean').dropna(how='all')
        if piv.shape[1] < 2 or piv.dropna().shape[0] < 3:
            axes[i].set_title(f"{strat}: insuficiente"); axes[i].axis("off"); continue
        cor = piv.corr()
        last_im = axes[i].imshow(cor.values, vmin=-1, vmax=1, aspect='equal')
        axes[i].set_xticks(range(cor.shape[1])); axes[i].set_xticklabels(cor.columns, rotation=30, ha='right', fontsize=8)
        axes[i].set_yticks(range(cor.shape[0])); axes[i].set_yticklabels(cor.index, fontsize=8)
        axes[i].set_title(f"Corr métricas — {strat}")
    if last_im is not None:
        cbar_ax = fig.add_axes([0.92, 0.15, 0.01, 0.7])
        fig.colorbar(last_im, cax=cbar_ax)
    plt.tight_layout(rect=[0,0,0.9,1]); plt.show()

for plant in PLANTS:
    print(f"=== {plant} ===")
    heatmap_drift_rate_inline(df_sum_all, plant)
    corr_matrices_inline(df_cols_all, plant)


## 5) Tablas: Top columnas (+ tipo) y percentiles

In [ ]:

def top_columns(df_cols, plant, metric, k=8):
    sub = df_cols.query("plant == @plant and metric == @metric").dropna(subset=['score'])
    if sub.empty: return pd.DataFrame()
    # incluimos tipo (numeric/categorical) si existe en df
    take_cols = [c for c in ['col','type','strategy','score'] if c in sub.columns]
    out = (sub[take_cols]
           .groupby(['col','type','strategy'], as_index=False)['score'].mean()
           .sort_values('score', ascending=False)
           .head(k))
    return out

def percentiles_p95(df_cols, plant):
    sub = df_cols.query("plant == @plant").dropna(subset=['score'])
    if sub.empty: return pd.DataFrame()
    grp = ['strategy','metric']
    p95 = (sub.groupby(grp, as_index=False)['score']
              .quantile(0.95).rename(columns={'score':'p95_score'}))
    p50 = (sub.groupby(grp, as_index=False)['score']
              .quantile(0.50).rename(columns={'score':'p50_score'}))
    g = p95.merge(p50, on=grp, how='left')
    g['stab'] = (1.0 - (g['p95_score'] - g['p50_score'])/g['p95_score'].replace(0,np.nan)).clip(0,1)
    return g

for plant in PLANTS:
    print(f"=== {plant} — Top columnas (ks/mannwhitney/psi/wasserstein) ===")
    for metric in ['ks','mannwhitney','psi','wasserstein']:
        t = top_columns(df_cols_all, plant, metric, k=6)
        if not t.empty:
            display(t.assign(metric=metric))
    print(f"=== {plant} — P95/P50 y estabilidad (stab) ===")
    g = percentiles_p95(df_cols_all, plant)
    display(g.sort_values(['strategy','metric']))


## 6) Selección (sensibilidad + concordancia + estabilidad)

In [ ]:

df_eval = df_sum_all.copy()
df_eval['SENS'] = np.minimum(df_eval['drift_rate_pct']/100.0, 0.85)

def mean_corr(df_cols, plant, strategy):
    sub = df_cols.query("plant == @plant and strategy == @strategy")
    if sub.empty: return np.nan
    piv = sub.pivot_table(index='col', columns='metric', values='score', aggfunc='mean').dropna(how='all')
    if piv.shape[1] < 2 or piv.dropna().shape[0] < 3: return np.nan
    cor = piv.corr()
    return float(cor.where(~np.eye(len(cor), dtype=bool)).stack().mean())

conc_rows = []
for plant in PLANTS:
    for strat in STRATEGIES:
        conc_rows.append({'plant': plant, 'strategy': strat, 'CONC': mean_corr(df_cols_all, plant, strat)})
df_conc = pd.DataFrame(conc_rows)

stab_rows = []
for plant in PLANTS:
    stab = percentiles_p95(df_cols_all, plant)
    if not stab.empty:
        stab_pl = stab.groupby(['strategy'], as_index=False)['stab'].mean().rename(columns={'stab':'STAB'})
        stab_pl['plant'] = plant
        stab_rows.append(stab_pl)
df_stab = pd.concat(stab_rows, ignore_index=True) if stab_rows else pd.DataFrame()

df_eval2 = (df_eval
            .groupby(['plant','strategy'], as_index=False)['SENS'].mean()
            .merge(df_conc, on=['plant','strategy'], how='left')
            .merge(df_stab, on=['plant','strategy'], how='left'))

df_eval2['CONC'] = df_eval2['CONC'].fillna(0.25).clip(lower=0)
df_eval2['STAB'] = df_eval2['STAB'].fillna(0.5).clip(0,1)

df_eval2['SELEC'] = 0.40*df_eval2['SENS'] + 0.40*df_eval2['CONC'] + 0.20*df_eval2['STAB']
display(df_eval2.sort_values(['plant','SELEC'], ascending=[True,False]))

best = (df_eval2.sort_values(['plant','SELEC'], ascending=[True,False])
                 .groupby('plant', as_index=False).first())
print("=== Recomendación por planta (estrategia prioritaria) ===")
display(best)
